In [1]:
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 
%matplotlib inline


In [7]:
df = pd.read_csv("8-fraud_detection.csv")
df.head()


,transaction_amount,transaction_risk_score,is_fraud
0,1.879910,-1.485035,0
1,0.377083,-2.238585,0
2,1.354312,-2.664638,0
3,-0.509843,-1.502950,0
4,0.863561,-1.906364,0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   transaction_amount      10000 non-null  float64
 1   transaction_risk_score  10000 non-null  float64
 2   is_fraud                10000 non-null  int64  
dtypes: float64(2), int64(1)
memory usage: 234.5 KB


In [4]:
from sklearn.model_selection import train_test_split ,GridSearchCV
from sklearn.linear_model import LogisticRegression 
from sklearn.metrics import accuracy_score,f1_score,classification_report,confusion_matrix

X = df.drop("is_fraud",axis=1 )
y = df["is_fraud"]

X_train ,X_test ,y_train ,y_test = train_test_split(X,y)
model = LogisticRegression()
model.fit(X_train,y_train)

y_pred = model.predict(X_test)

print(confusion_matrix(y_test,y_pred))






[[2459    0]
 [  33    8]]


In [21]:
params= {
    "C": [0.1,2,7,12,42,90],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'lbfgs'] # lbfgs hates l1!
}

grid_search= GridSearchCV(estimator=model,param_grid=params,cv=4,scoring="precision")
grid_search.fit(X_train,y_train)
print(grid_search.best_params_)
print(grid_search.best_score_)
y_pred=grid_search.predict(X_test)
confusion_matrix(y_test, y_pred)

/opt/anaconda3/envs/youtube_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}
0.95


/opt/anaconda3/envs/youtube_env/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
24 fits failed out of a total of 96.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
24 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/anaconda3/envs/youtube_env/lib/python3.10/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/opt/anaconda3/envs/youtube_env/lib/python3.10/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/opt/anaconda3/envs/youtube_env/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py", line 1218,

array([[2458,    1],
       [  34,    7]])

In [20]:
df["is_fraud"].value_counts()

is_fraud
0    9846
1     154
Name: count, dtype: int64

**Rule:**  
For classification → always use **StratifiedKFold**  
For regression → use **KFold**  
For time series → use **TimeSeriesSplit**

---

## 6) Which scoring should I choose? (imbalanced case)

- Precision → when false alarms (FP) are expensive  
- Recall → when missing positives (FN) is expensive  
- F1 → balance between precision and recall  
- Average Precision (PR-AUC) → very good for imbalanced problems (fraud)

---

## 7) Clean GridSearch idea (no bad combinations)

In [22]:
#hyperparameter tuning with class weights to handle imbalance
penalty=['l1', 'l2', 'elasticnet']
c_values=[100,10,1.0,0.1,0.01]
solver=['newton-cg', 'lbfgs', 'liblinear', 'sag', 'saga']
class_weight=[{0:w,1:y} for w in [1,10,50,100] for y in [1,10,50,100]]

In [23]:
class_weight

[{0: 1, 1: 1},
 {0: 1, 1: 10},
 {0: 1, 1: 50},
 {0: 1, 1: 100},
 {0: 10, 1: 1},
 {0: 10, 1: 10},
 {0: 10, 1: 50},
 {0: 10, 1: 100},
 {0: 50, 1: 1},
 {0: 50, 1: 10},
 {0: 50, 1: 50},
 {0: 50, 1: 100},
 {0: 100, 1: 1},
 {0: 100, 1: 10},
 {0: 100, 1: 50},
 {0: 100, 1: 100}]

In [25]:
import warnings
warnings.filterwarnings('ignore')
grid_search.fit(X_train,y_train)

,estimator,LogisticRegression()
,param_grid,"{'C': [0.1, 2, ...], 'penalty': ['l1', 'l2'], 'solver': ['liblinear', 'lbfgs']}"
,scoring,'precision'
,n_jobs,None
,refit,True
,cv,4
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l1'


In [31]:
grid_search.best_params_

{'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}

In [33]:
y_pred=grid_search.predict(X_test)

In [34]:
y_pred

array([0, 0, 0, ..., 0, 0, 0], shape=(2500,))

In [35]:
score=accuracy_score(y_pred,y_test)
print("score: ", score)
print(classification_report(y_pred,y_test))
print("confusion matrix: \n " , confusion_matrix(y_pred,y_test))

score:  0.986
              precision    recall  f1-score   support

           0       1.00      0.99      0.99      2492
           1       0.17      0.88      0.29         8

    accuracy                           0.99      2500
   macro avg       0.59      0.93      0.64      2500
weighted avg       1.00      0.99      0.99      2500

confusion matrix: 
  [[2458   34]
 [   1    7]]


In [36]:
grid_search.best_params_

{'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}

# 🚨 Midas Security Simulation: Catching the Invisible Thief

## 1. The Scenario (Senaryo) 🏦
**Şirket:** Midas (Yatırım & Borsa Uygulaması)
**Problem:** Çalıntı kredi kartlarıyla hisse senedi almaya çalışan dolandırıcılar var.
**Veri Durumu:** Günlük 10.000 işlem yapılıyor. Bunların sadece **%1'i (100 tanesi)** dolandırıcılık (Fraud). Geri kalan %99'u temiz.

**Hedef:**
* **Accuracy (Doğruluk) ÖNEMSİZ:** Eğer model "Herkes Temiz" derse %99 doğru çıkar ama banka batar.
* **Recall (Duyarlılık) KRİTİK:** O %1'lik hırsız kesimin tamamını yakalamak zorundayız.

---

## 2. The Simulation Code (Python) 🐍
Bu kod, iki farklı yaklaşımı yarıştırır:
1.  **Lazy Model:** Hiçbir ayar yapılmamış, "Accuracy" odaklı standart model.
2.  **Midas Guard (Tuned Model):** Dengesiz veri için özel ayarlanmış (`class_weight='balanced'`), hırsız avcısı model.

```python
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# ==========================================
# 1. VERİ SİMÜLASYONU (Imbalanced Data Creation)
# ==========================================
# 10,000 İşlem yaratıyoruz
n_samples = 10000
n_fraud = 100  # Sadece %1 Fraud

# Temiz İşlemler (Normal dağılım, düşük tutarlar, güvenli cihazlar)
X_legit = np.random.normal(loc=[50, 10], scale=[20, 2], size=(n_samples - n_fraud, 2))
y_legit = np.zeros(n_samples - n_fraud)

# Fraud İşlemler (Yüksek tutarlar, güvensiz cihaz skorları)
X_fraud = np.random.normal(loc=[500, 2], scale=[100, 5], size=(n_fraud, 2))
y_fraud = np.ones(n_fraud)

# Veriyi Birleştir
X = np.vstack((X_legit, X_fraud))
y = np.hstack((y_legit, y_fraud))

# Kolon İsimleri: [İşlem Tutarı ($), Cihaz Güven Skoru (0-10)]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"📊 Test Setindeki Toplam Fraud Sayısı: {sum(y_test)}")
print("="*40)

# ==========================================
# 2. MODEL 1: THE "LAZY" MODEL (Standart)
# ==========================================
# Hiçbir tuning yok. Standart Lojistik Regresyon.
lazy_model = LogisticRegression()
lazy_model.fit(X_train, y_train)
y_pred_lazy = lazy_model.predict(X_test)

print("\n❌ MODEL 1 (Standart) Sonuçları:")
# Hırsızların kaçını yakaladı?
print(f"Yakalanan Hırsız Oranı (Recall): {classification_report(y_test, y_pred_lazy, output_dict=True)['1.0']['recall']:.2f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lazy))

# ==========================================
# 3. MODEL 2: MIDAS GUARD (Tuned for Imbalance) 🛡️
# ==========================================
# class_weight='balanced': Azınlık sınıfına (Fraud) süper güç verir.
# C=0.01: Modeli biraz esnetir (Regularization) ki ezber yapmasın.
midas_guard = LogisticRegression(class_weight='balanced', C=0.01, solver='liblinear')
midas_guard.fit(X_train, y_train)
y_pred_guard = midas_guard.predict(X_test)

print("\n✅ MODEL 2 (Tuned - Midas Guard) Sonuçları:")
print(f"Yakalanan Hırsız Oranı (Recall): {classification_report(y_test, y_pred_guard, output_dict=True)['1.0']['recall']:.2f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_guard))

In [40]:
Yakalanan Hırsız Oranı (Recall): 0.35  (Çok Kötü!)
Confusion Matrix:
[[1980    0]   <-- Temizleri bildi (Harika!)
 [  13    7]]  <-- 20 Hırsızın 13'ünü KAÇIRDI! (Felaket)

SyntaxError: unterminated string literal (detected at line 4) (2914969577.py, line 4)

In [ ]:
Yakalanan Hırsız Oranı (Recall): 0.95  (Harika!)
Confusion Matrix:
[[1900   80]   <-- 80 Temiz müşteriyi yanlışlıkla şüpheli sandı (False Positive).
 [   1   19]]  <-- 20 Hırsızın 19'unu YAKALADI! 🚨